# Importation Librairie

In [1]:
import zlib
import hashlib
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

# DQN

## V1

In [5]:
class Compressor:
    def __init__(self, method='zlib'):
        self.method = method

    def compress(self, data: bytes) -> bytes:
        if self.method == 'zlib':
            return zlib.compress(data)
        # on pourra rajouter d'autres méthodes plus tard
        raise NotImplementedError

    def decompress(self, data: bytes) -> bytes:
        if self.method == 'zlib':
            return zlib.decompress(data)
        raise NotImplementedError

    @staticmethod
    def checksum(data: bytes) -> str:
        return hashlib.sha256(data).hexdigest()

    def verify(self, original: bytes, compressed: bytes) -> bool:
        decompressed = self.decompress(compressed)
        return self.checksum(original) == self.checksum(decompressed)

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, action_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon=1.0):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon
        self.memory = deque(maxlen=10000)
        self.model = DQN(state_dim, action_dim)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim-1)
        state = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            q_values = self.model(state)
        return q_values.argmax().item()

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def replay(self, batch_size=32):
        if len(self.memory) < batch_size:
            return
        batch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in batch:
            state_t = torch.FloatTensor(state).unsqueeze(0)
            next_state_t = torch.FloatTensor(next_state).unsqueeze(0)
            q_target = reward
            if not done:
                q_target += self.gamma * self.model(next_state_t).max().item()
            q_values = self.model(state_t)
            # conversion du target en float
            loss = self.loss_fn(q_values[0][action], torch.tensor(q_target, dtype=torch.float))
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()


## Entrainement

In [19]:
DATA_DIR = 'Data'
RESULT_DIR = 'Result'
BATCH_SIZE = 32
EPOCHS = 1000

# Initialisation
compressor = Compressor()
agent = DQNAgent(state_dim=100, action_dim=5)  # exemple, à adapter selon ton action space
best_sizes = {}  # dictionnaire pour stocker la meilleure compression par fichier

# Récupère tous les fichiers dans Data et sous-dossiers
filepaths = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        full_path = os.path.join(root, file)
        filepaths.append(full_path)

print(f"[INFO] Nombre total de fichiers : {len(filepaths)}")

for epoch in range(EPOCHS):
    print(f"\n[INFO] Epoch {epoch+1}/{EPOCHS}, epsilon={agent.epsilon:.3f}")

    for filepath in filepaths:
        # Lecture du fichier
        with open(filepath, 'rb') as f:
            data = f.read()

        # Création de l'état : histogramme normalisé des bytes
        state = np.histogram(np.frombuffer(data, dtype=np.uint8), bins=100, range=(0,255))[0].astype(float)/len(data)

        # Choix de l'action par le DQN
        action = agent.act(state)

        # Compression (adapter selon l'action)
        compressed_data = compressor.compress(data)  # ici on pourrait moduler action

        # Vérification lossless
        reward = 0
        done = False
        if compressor.verify(data, compressed_data):
            done = True
            prev_best = best_sizes.get(filepath, len(data)+1)
            # Reward basé sur l'amélioration de la compression
            reward = max(0, prev_best - len(compressed_data))
            # Sauvegarde uniquement si meilleure compression
            if len(compressed_data) < prev_best:
                rel_path = os.path.relpath(filepath, DATA_DIR)
                result_path = os.path.join(RESULT_DIR, rel_path + '.bin')
                os.makedirs(os.path.dirname(result_path), exist_ok=True)
                with open(result_path, 'wb') as f:
                    f.write(compressed_data)
                best_sizes[filepath] = len(compressed_data)
                print(f"[INFO] Nouvelle meilleure compression pour {rel_path}: {len(compressed_data)} bytes")

        # État suivant (ici on garde le même histogramme pour simplifier)
        next_state = state

        # Stockage dans la mémoire et apprentissage
        agent.remember(state, action, reward, next_state, done)
        agent.replay(batch_size=BATCH_SIZE)

    # Décroissance epsilon pour exploration progressive
    agent.epsilon = max(0.1, agent.epsilon * 0.995)

[INFO] Nombre total de fichiers : 1104

[INFO] Epoch 1/1000, epsilon=1.000
[INFO] Nouvelle meilleure compression pour weather_prediction_dataset.csv: 888522 bytes
[INFO] Nouvelle meilleure compression pour Texte/RSA4096.txt: 2476 bytes
[INFO] Nouvelle meilleure compression pour Texte/Q-Learning.txt: 6469 bytes
[INFO] Nouvelle meilleure compression pour Texte/archive(2)/historical/historical_76.txt: 2360 bytes
[INFO] Nouvelle meilleure compression pour Texte/archive(2)/historical/historical_5.txt: 810 bytes
[INFO] Nouvelle meilleure compression pour Texte/archive(2)/historical/historical_2.txt: 2854 bytes
[INFO] Nouvelle meilleure compression pour Texte/archive(2)/historical/historical_18.txt: 664 bytes
[INFO] Nouvelle meilleure compression pour Texte/archive(2)/historical/historical_50.txt: 5665 bytes
[INFO] Nouvelle meilleure compression pour Texte/archive(2)/historical/historical_25.txt: 513 bytes
[INFO] Nouvelle meilleure compression pour Texte/archive(2)/historical/historical_98.tx

KeyboardInterrupt: 